<a href="https://colab.research.google.com/github/Perine23Dev/analyse-entreprises-emploi-republique-du-congo/blob/feature%2Fexploration-donnees/projet_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Perine23Dev/analyse-entreprises-emploi-republique-du-congo/blob/exploration_rosy/projet_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importation

In [49]:
import io
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
creds, _ = default()
service = build("drive", "v3", credentials=creds)

file_id = "18zTTJ434DgPIt9JCoXujQ6HgvXpY0zKg"

requete = service.files().get_media(fileId=file_id)
tampon = io.BytesIO()
telechargement = MediaIoBaseDownload(tampon, requete)

fini = False
while not fini:
    _, fini = telechargement.next_chunk()

tampon.seek(0)
df = pd.read_excel(tampon)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


**Exploration de base**

In [50]:
dimension=df.shape
print("la dimension du data frame est : ", dimension)

la dimension du data frame est :  (18787, 20)


Conclusion : ce jeu de données contient 18787 lignes et 20 colonnes

In [51]:
#affichage des 5 premières lignes du dataframe
df.head()

,Année création,Département,Raison sociale établissement principal,Activité déclarée,Activité au sens de la nomenclature OHADA,Forme juridique,Statut juridique,Catégorie,Genre,Age,Nationalité,Branche,Secteur,capital,part_capital_etranger,salarie,nbre_empl_congolais,chiffre_aff_mo_annuel,chiffre_aff_mo_annuel_1,chiffre_aff_mo_annuel_2
0,2023,Pointe-Noire,EL SHADDAI,MERCERIE,Commerce de détail en magasin spécialisé de te...,EI,Entreprise individuelle,TPE-PE,Femme,45.0,Congolaise,Commerce,Tertiaire,500000,0.0,1.0,1.0,1000000,1500000,2000000.0
1,2023,Niari,BADJEL,Commerce de détail en magasin spécialisé des p...,Commerce de détail en magasin spécialisé des p...,EI,Entreprise individuelle,TPE,Homme,49.0,Mauritanienne,Commerce,Tertiaire,NaN,NaN,1.0,0.0,1500000,2000000,2500000.0
2,2015,Pointe-Noire,HIRAM SERVICES,MISE A DISPOSITION DE PERSONNEL,activités liées aux ressources humaines,SARLU,Société de personnes,TPE-PE,Homme,43.0,Congolaise,Services aux entreprises,Tertiaire,0,0.0,1.0,1.0,0,0,0.0
3,2015,Pointe-Noire,HB,VENTE D'ELECTROMENAGER,commerce de détail en magasin spécialisé d’art...,EI,Entreprise individuelle,TPE-PE,Homme,40.0,Malienne,Commerce,Tertiaire,0,0.0,4.0,0.0,0,0,0.0
4,2026,Niari,BADJEL,Commerce de détail en magasin spécialisé de pr...,Commerce de détail en magasin spécialisé de pr...,EI,Entreprise individuelle,TPE,Homme,49.0,Mauritanienne,Commerce,Tertiaire,NaN,NaN,1.0,0.0,1500000,2000000,2500000.0


Les cinq premières lignes montrent une diversité d’entreprises en termes d’année de création, d’activité, de forme juridique, de nationalité et de taille. Les entreprises observées appartiennent principalement au secteur tertiaire, notamment au commerce et aux services. On remarque également la présence de valeurs manquantes dans certaines variables, notamment le capital et la part de capital étranger. La raison sociale BADJEL apparaît deux fois, ce qui devra être vérifié lors de l’analyse des doublons. Enfin, les données contiennent des informations sur les salariés et le chiffre d’affaires permettant d’étudier ultérieurement la situation et l’évolution des entreprises.

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18787 entries, 0 to 18786
Data columns (total 20 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Année création                             18787 non-null  int64  
 1   Département                                18787 non-null  object 
 2   Raison sociale établissement principal     18787 non-null  object 
 3   Activité déclarée                          18777 non-null  object 
 4   Activité au sens de la nomenclature OHADA  18683 non-null  object 
 5   Forme juridique                            18787 non-null  object 
 6   Statut juridique                           18787 non-null  object 
 7   Catégorie                                  5545 non-null   object 
 8   Genre                                      18675 non-null  object 
 9   Age                                        18455 non-null  float64
 10  Nationalité           

La base de données comprend 18 787 observations et 20 variables, couvrant les caractéristiques démographiques, juridiques et économiques des entreprises. Elle contient 13 variables qualitatives et 7 variables quantitatives. La plupart des variables sont correctement renseignées, mais certaines présentent des données manquantes, notamment la variable Catégorie, dont environ 70,5 % des valeurs sont absentes. Les variables capital et chiffre d'affaires nécessitent également une vérification de leur format, car elles sont enregistrées comme variables textuelles alors qu'elles représentent a priori des grandeurs quantitatives. Ces éléments devront être traités avant l'analyse statistique.

In [53]:
#affichage des premières statistiques du dataframes
df.describe()

,Année création,Age,part_capital_etranger,salarie,nbre_empl_congolais,chiffre_aff_mo_annuel_1,chiffre_aff_mo_annuel_2
count,18787.000000,18455.000000,1.713600e+04,18784.000000,18724.000000,1.878700e+04,1.878700e+04
mean,2023.647948,40.550257,2.230160e+06,21.979291,2.457862,8.991517e+07,9.798952e+07
std,3.398257,21.559952,1.480841e+08,2614.726019,13.735897,5.934576e+09,6.022792e+09
min,1949.000000,-174.000000,0.000000e+00,0.000000,-2.000000,0.000000e+00,0.000000e+00
25%,2023.000000,32.000000,0.000000e+00,1.000000,1.000000,1.000000e+06,1.000000e+06
50%,2025.000000,38.000000,0.000000e+00,1.000000,1.000000,2.500000e+06,3.000000e+06
75%,2025.000000,47.000000,0.000000e+00,2.000000,2.000000,1.000000e+07,1.000000e+07
max,2028.000000,1043.000000,1.269362e+10,358358.000000,536.000000,8.000000e+11,8.100000e+11


Le tableau des statistiques descriptives montre que les variables quantitatives présentent des niveaux et des dispersions différents. L'âge moyen est de 40,55 ans, avec une médiane de 38 ans et un intervalle interquartile compris entre 32 et 47 ans. Le nombre de salariés présente une moyenne de 21,98 contre une médiane de 1, tandis que 75 % des observations comptent au maximum 2 salariés. Cette différence entre la moyenne et la médiane traduit une importante dispersion des effectifs. Concernant le chiffre d'affaires, la moyenne est de 89,92 millions pour la première période et de 97,99 millions pour la seconde, alors que les médianes sont respectivement de 2,5 millions et 3 millions. L'écart entre ces valeurs indique également une forte dispersion du chiffre d'affaires. Enfin, certaines valeurs minimales et maximales, notamment pour l'âge, le nombre d'employés et le chiffre d'affaires, apparaissent particulièrement élevées ou incohérentes et devront être vérifiées.

In [54]:
#verifications des valeurs manquantes sur l'ensembler du dataframes
df.isnull().sum()

,0
Année création,0
Département,0
Raison sociale établissement principal,0
Activité déclarée,10
Activité au sens de la nomenclature OHADA,104
Forme juridique,0
Statut juridique,0
Catégorie,13242
Genre,112
Age,332


In [55]:
#calcul du pourcentage des valeurs manquantes
(df.isnull().sum() / len(df) * 100).round(2)


,0
Année création,0.00
Département,0.00
Raison sociale établissement principal,0.00
Activité déclarée,0.05
Activité au sens de la nomenclature OHADA,0.55
Forme juridique,0.00
Statut juridique,0.00
Catégorie,70.48
Genre,0.60
Age,1.77


L’analyse des valeurs manquantes montre que la majorité des variables sont bien renseignées. La variable « Catégorie » présente toutefois un taux élevé de valeurs manquantes (70,48 %), suivie du capital (10,82 %) et de la part du capital étranger (8,79 %). Les autres variables présentent des taux de valeurs manquantes faibles, généralement inférieurs à 3 %. Ces variables devront donc faire l’objet d’une attention particulière lors du nettoyage des données.




In [56]:
df.duplicated().sum()

np.int64(0)

Aucune ligne n'est répétée, donc les données ne contiennent pas de doublons.

**Exploration des 4 premières variables**

Cette partie porte sur l'exploration des variables suivantes :
- Année création
- Département
- Raison sociale établissement principal
- Activité déclarée

L'objectif est d'identifier les principales caractéristiques, les modalités
dominantes et les éventuelles particularités de ces variables.

In [57]:
#affichage des premieres valeurs des premères 4 colonnes
df.iloc[:, 0:4].head()

,Année création,Département,Raison sociale établissement principal,Activité déclarée
0,2023,Pointe-Noire,EL SHADDAI,MERCERIE
1,2023,Niari,BADJEL,Commerce de détail en magasin spécialisé des p...
2,2015,Pointe-Noire,HIRAM SERVICES,MISE A DISPOSITION DE PERSONNEL
3,2015,Pointe-Noire,HB,VENTE D'ELECTROMENAGER
4,2026,Niari,BADJEL,Commerce de détail en magasin spécialisé de pr...


Les premières lignes présentent des établissements créés entre 2015 et 2026, situés notamment dans les départements de Pointe-Noire et du Niari. Les raisons sociales sont différentes, avec toutefois une répétition de “BADJEL”. Les activités déclarées sont variées et certaines sont affichées sous forme de texte tronqué.
Cet aperçu permet également de relever une première question concernant la qualité des données textuelles : certaines valeurs sont écrites en majuscules et les formats peuvent être hétérogènes. Cette situation devra être vérifiée sur l’ensemble du jeu de données avant d’envisager une éventuelle standardisation.

In [58]:
df.iloc[:, :4][df.iloc[:, :4].isna().any(axis=1)]

,Année création,Département,Raison sociale établissement principal,Activité déclarée
10236,2009,Pointe-noire,KAYE - SHOP,NaN
10298,2021,Pointe-noire,K.TRANSPORTS CONGO,NaN
10361,2022,Pointe-noire,PERE EMILE BIABIEGNY,NaN
10371,2022,Brazzaville,ETS NJK GROUP,NaN
10414,2022,Brazzaville,OSYS REAL ESTATE,NaN
10679,2022,Brazzaville,FONDATION MWETI INGRID,NaN
10745,1994,Brazzaville,OFFICE NOTARIAL MAITRE ESTHER NANETTE NOTE,NaN
10780,2022,Pointe-noire,ETS NGOUDISS,NaN
15992,2026,Pointe-noire,DU CIEL OPTIQUE,NaN
16073,2026,Pointe-noire,INTERNATIONAL TRADE BUSINESS AND SERVICES-BATÎ...,NaN


**Exploration de l'année de création**

In [59]:
df["Année création"].value_counts().sort_index()

,count
Année création,
1949,1
1959,1
1962,1
1989,1
1992,1
1993,2
1994,2
1995,3
1996,1


**La variable “Année création”** présente des effectifs différents selon les années. Certaines années comptent une seule observation, tandis que d’autres en comptent plusieurs centaines. Les valeurs observées s’étendent de 1949 à 2028. Pour la variable “Département”, on observe également des modalités présentant des différences de casse, notamment “Pointe-Noire” et “Pointe-noire”, ainsi que “Cuvette-Ouest”, “Cuvette-ouest” et “CUVETTE”.

In [60]:
df['Année création'].describe()

,Année création
count,18787.000000
mean,2023.647948
std,3.398257
min,1949.000000
25%,2023.000000
50%,2025.000000
75%,2025.000000
max,2028.000000


La variable “Année création” contient 18 787 observations. Les valeurs observées vont de 1949 à 2028. La moyenne est de 2023,65, tandis que la médiane est de 2025. Les quartiles indiquent une valeur de 2023 au premier quartile et de 2025 au troisième quartile.

# **Verification des différentes années de créations**

In [61]:
pd.DataFrame(
    sorted(df["Année création"].unique()),
    columns=["Année création"]
)

,Année création
0,1949
1,1959
2,1962
3,1989
4,1992
5,1993
6,1994
7,1995
8,1996
9,1997


La colonne “Année création” présente des années allant de 1949 à 2028.

# **Exploration de la colonne département**

In [62]:
df["Département"].value_counts()

,count
Département,
Brazzaville,9879
Pointe-noire,4873
Pointe-Noire,2385
Sangha,427
Niari,417
Cuvette,256
Bouenza,208
Pool,148
Kouilou,71


La variable “Département” présente une forte concentration des établissements à Brazzaville, suivie de Pointe-Noire. On remarque également des différences d’écriture pour certains départements, comme “Pointe-noire” et “Pointe-Noire”.

**Exploration des différents départements**

In [63]:
df["Département"].unique()

array(['Pointe-Noire', 'Niari', 'Pointe-noire', 'Brazzaville', 'Bouenza',
       'Sangha', 'Likouala', 'Cuvette-Ouest', 'Cuvette', 'Pool',
       'Kouilou', 'Plateaux', 'Lékoumou', 'Djoue-Lefini', 'Cuvette-ouest',
       'Nkeni-Alima', 'CUVETTE', 'Owando', 'Nkayi', 'Congo-Oubangui'],
      dtype=object)

**“Département”** contient 20 modalités distinctes. On observe toutefois des incohérences dans l’écriture de certaines modalités, notamment “Pointe-Noire” et “Pointe-noire”, ainsi que “Cuvette-Ouest” et “Cuvette-ouest”. La valeur “CUVETTE” apparaît également en majuscules. Ces variations peuvent correspondre à un même département et devront être standardisées lors du nettoyage des données.

**Exploration de la variable Raison social**

In [64]:
df["Raison sociale établissement principal"].value_counts().head(20)

,count
Raison sociale établissement principal,
DIEU MERCI,13
M.K,12
M.D,10
S.M,9
MS,9
GRACE DIVINE,8
M.S,8
IB,8
LA GRACE DIVINE,7


**La variable “Raison sociale établissement principal”**, qui correspond au nom de l’entreprise ou de l’établissement, présente des effectifs différents selon les raisons sociales. Certaines apparaissent plusieurs fois, comme “S.M”, “MS”, “GRACE DIVINE” et “M.S”. On observe également des différences de casse dans certaines raisons sociales, par exemple “ETS DIEU MERCI” et “Dieu merci”.

Verification des espaces espaces au début ou à la fin des données colonnes textes

In [65]:
colonnes_texte = [
    "Département",
    "Raison sociale établissement principal",
    "Activité déclarée"
]

df[colonnes_texte].apply(
    lambda col: col.astype(str).str.strip().ne(col.astype(str))
).sum()

,0
Département,0
Raison sociale établissement principal,128
Activité déclarée,202


Excepté la colonne département , les colonnes Raison social et activité déclaré contiennent les espaces en debut et à la fin des textes

**identification les cellules textuelles vides ou contenant uniquement des espaces dans les variables textuelles.**

In [66]:
df[colonnes_texte].apply(
    lambda col: col.astype(str).str.strip().eq("")
).sum()

,0
Département,0
Raison sociale établissement principal,0
Activité déclarée,0


Il n' y a pas d'espace entre les textes

**Sythese : **L’exploration des données a permis d’identifier la structure générale du jeu de données et d’examiner les principales caractéristiques des quatre premières variables : Année création, Département, Raison sociale établissement principal et Activité déclarée.
La variable Année création contient 18 787 observations, avec des valeurs comprises entre 1949 et 2028. Sa moyenne est de 2023,65 et sa médiane de 2025.
La variable Département comporte 20 modalités distinctes. Certaines modalités présentent des différences de casse ou d’écriture, notamment « Pointe-Noire » / « Pointe-noire », « Cuvette-Ouest » / « Cuvette-ouest » et « CUVETTE ».
La variable Raison sociale établissement principal contient 17 603 valeurs distinctes sur 18 787 observations. Certaines raisons sociales sont répétées et des différences de formatage ont été observées, notamment la présence d’espaces superflus et d’espaces insécables.
Enfin, la variable Activité déclarée contient 18 777 valeurs renseignées, correspondant à 2 798 activités distinctes. L’activité la plus fréquente apparaît 1 405 fois, et 10 valeurs manquantes ont été relevées.
Cette exploration a ainsi permis de relever les effectifs, les modalités, les valeurs extrêmes, les répétitions, les valeurs manquantes et certaines irrégularités de formatage qui pourront être prises en compte lors de l’étape de nettoyage.

**Exploration des variables 5 à 8**

Activité au sens de la nomenclature OHADA, Forme juridique, Statut juridique, Catégorie.

In [67]:
segment = df.iloc[:, 4:4+4]
segment.head(5)
# Segmenter les 4 colonnes suivantes pour avoir un plan de travail plus reduit sanss appliquer une division avec le df

,Activité au sens de la nomenclature OHADA,Forme juridique,Statut juridique,Catégorie
0,Commerce de détail en magasin spécialisé de te...,EI,Entreprise individuelle,TPE-PE
1,Commerce de détail en magasin spécialisé des p...,EI,Entreprise individuelle,TPE
2,activités liées aux ressources humaines,SARLU,Société de personnes,TPE-PE
3,commerce de détail en magasin spécialisé d’art...,EI,Entreprise individuelle,TPE-PE
4,Commerce de détail en magasin spécialisé de pr...,EI,Entreprise individuelle,TPE


In [68]:
# verification des espaces en deebut et fin de chaques cellules
colonnes = [
   	"Activité au sens de la nomenclature OHADA",
    "Forme juridique",
    "Statut juridique",
    "Catégorie"
]

df[colonnes].apply(
    lambda col: col.astype(str).str.strip().ne(col.astype(str))
).sum()

,0
Activité au sens de la nomenclature OHADA,244
Forme juridique,3
Statut juridique,0
Catégorie,0


Exploration de la colonne Activité au sens... OHADA

In [69]:
df['Activité au sens de la nomenclature OHADA'].value_counts()

,count
Activité au sens de la nomenclature OHADA,
"Commerce de détail en magasin spécialisé des produits alimentaires, boissons et tabacs",1421
"Commerce de détail en magasin spécialisé de textiles, habillement, chaussures et articles en cuir",770
"commerce de détail en magasin spécialisé des produits alimentaires, boissons et tabacs",756
Commerce de détail en magasin non spécialisé,465
"Commerce de détail en magasin spécialisé de quincaillerie, peintures, verre, tapis et revêtement de sols et murs",441
...,...
Activités des sièges sociaux ; conseil en gestion (Assistance et Conseil),1
Commerce de détail sur éventaires et marchés d'articles non alimentaires,1
Fabrication des corps gras d'origine animale et végétale,1


Exploration du statut juridique

In [70]:
df['Statut juridique'].value_counts()

,count
Statut juridique,
Entreprise individuelle,13318
Société de personnes,4906
Société de capitaux,563


Exploration de la colonne categorie

In [71]:
df['Catégorie'].value_counts()

,count
Catégorie,
TPE-PE,3310
TPE,1760
PE,382
ME-GE,67
ME,25
GE,1


Partie d'exploration qui concerne les colonnes : **Genre**, **Age**, **Nationalité**, **Branche**

In [72]:
# Sélection sécurisée avec conservation de l'index/ID
df_dirigeants = df[['Genre', 'Age', 'Nationalité', 'Branche']].copy()

In [73]:
# stockage du nombre de valeurs nulles pour chaque colonne ('Genre', 'Age', 'Nationalité', 'Branche')
count_genre_nan = df_dirigeants['Genre'].isnull().sum()
count_age_nan = df_dirigeants['Age'].isnull().sum()
count_nationalite_nan = df_dirigeants['Nationalité'].isnull().sum()
count_branche_nan = df_dirigeants['Branche'].isnull().sum()

# affichage du rapport par colonne
print("Rapport des valeurs nulles par colonne :")
print("Genre : ", count_genre_nan)
print("Age : ", count_age_nan)
print("Nationalité : ", count_nationalite_nan)
print("Branche : ", count_branche_nan)

Rapport des valeurs nulles par colonne :
Genre :  112
Age :  332
Nationalité :  448
Branche :  26


In [74]:
# 1. Inspection rapide
df_dirigeants.info()

# 2. Analyse de l'Âge (Numérique)
print("Statistiques de l'Âge :")
print(df_dirigeants['Age'].describe())

# 3. Distribution du Genre, Nationalité et Branche (Catégorielles)
print("\nRépartition par Genre :")
print(df_dirigeants['Genre'].value_counts(dropna=False, normalize=True) * 100)

print("\nTop 5 des Nationalités :")
print(df_dirigeants['Nationalité'].value_counts().head(5))

# 4. Croisement Clé (ex: Âge moyen par Genre, ou Genre par Branche)
print(df_dirigeants.groupby('Genre')['Age'].agg(['mean', 'median', 'count']))
print(pd.crosstab(df_dirigeants['Branche'], df_dirigeants['Genre'], normalize='index') * 100)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18787 entries, 0 to 18786
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Genre        18675 non-null  object 
 1   Age          18455 non-null  float64
 2   Nationalité  18339 non-null  object 
 3   Branche      18761 non-null  object 
dtypes: float64(1), object(3)
memory usage: 587.2+ KB
Statistiques de l'Âge :
count    18455.000000
mean        40.550257
std         21.559952
min       -174.000000
25%         32.000000
50%         38.000000
75%         47.000000
max       1043.000000
Name: Age, dtype: float64

Répartition par Genre :
Genre
Masculin    39.463459
Homme       34.997605
Femme       20.391760
Feminin      4.551019
NaN          0.596157
Name: proportion, dtype: float64

Top 5 des Nationalités :
Nationalité
Congolaise       13552
Malienne          1780
Chinoise           373
Camerounaise       348
Mauritanienne      286
Name: count, dtype: int64
       

# 📝 Bilan de l'exploration initiale : Profil des Dirigeants (`Genre`, `Age`, `Nationalité`, `Branche`)

---

## 1. Constat sur la Qualité des Données & Actions de Nettoyage Requis

L'inspection initiale de ces 4 variables met en évidence plusieurs anomalies majeures qui nécessitent une étape de prétraitement avant toute analyse définitive :

* **Valeurs aberrantes sur l'Âge :** Présence de valeurs impossibles (minimum à `-174 ans` et maximum à `1043 ans`).
  * **Action :** Filtrer la variable pour ne conserver que la plage réaliste $[18, 90]$ ans.
* **Redondance de saisie sur le Genre :** Coexistence de doublons sémantiques (`Homme`/`Masculin` et `Femme`/`Feminin`).
  * **Action :** Harmoniser les modalités pour obtenir une variable binaire propre.
* **Doublons de saisie dans les Branches :** Présence d'espaces superflus ou de légères variations de casse créant des doublons de catégories (ex. *Activités spécialisées, scientifiques...*).
  * **Action :** Nettoyer les chaînes de caractères (`.str.strip()`).

---

## 2. Premiers Insights Métiers

### 👤 Représentation par Genre
Après regroupement des catégories, le tissu entrepreneurial est très majoritairement masculin : environ **74,5 % d'hommes** contre **25,0 % de femmes** (avec ~0,6 % de valeurs manquantes).

### 🎂 Distribution de l'Âge
Bien que la moyenne soit actuellement faussée par les outliers, la **médiane à 38 ans** (avec un intervalle interquartile compris entre 32 et 47 ans) indique une population d'entrepreneurs relativement jeune et en pleine activité professionnelle.

### 🌍 Origine Nationale
La nationalité **Congolaise** est très largement dominante (**~73,8 %** des entreprises renseignées, soit 13 552 établissements). Parmi les ressortissants étrangers, la communauté **Malienne** constitue le premier groupe d'entrepreneurs (1 780 entreprises), suivie de la Chine, du Cameroun et de la Mauritanie.

### 📊 Secteurs d'Activité et Genre (Analyse Croisée)
* **Secteurs à forte présence féminine :** Les femmes sont principalement représentées dans les *Autres activités de services* (~54 %) et l'*Hébergement/Restauration* (~43 %).
* **Secteurs fortement masculinisés :** Les domaines de la *Construction*, des *Transports*, de l'*Industrie* et des *Activités extractives* restent dominés à plus de 80-90 % par des hommes.


Partie C
---



In [75]:
seg_2 = df.iloc[:, 12:12+4]
seg_2.head(3)

,Secteur,capital,part_capital_etranger,salarie
0,Tertiaire,500000,0.0,1.0
1,Tertiaire,NaN,NaN,1.0
2,Tertiaire,0,0.0,1.0


In [76]:
df['salarie'] == 0

,salarie
0,False
1,False
2,False
3,False
4,False
...,...
18782,False
18783,False
18784,False
18785,False


In [77]:
df['Secteur'].value_counts()
# Des espaces remarqués dans la sasie des types de secteur
#Le secteur tertiaire arrive en tete avec 9450 entreprise qui n'emploient qu'un seul employé

,count
Secteur,
Tertiaire,16087
Secondaire,2008
Primaire,672
secondaire,1
Tertiaire,1


In [78]:
df['capital'].value_counts(ascending = True)

,count
capital,
48787260,1
450000,1
A,1
200000,1
83179477792,1
...,...
2000000,113
5000000,241
10000000,362


In [79]:
capital_num = pd.to_numeric(df['capital'], errors= "coerce")
(capital_num < 0).sum()

np.int64(0)

Aucune valeur negative observée mais des valeurs incoherences de types str que nous allons nettoyer dans la prochaine phase

In [80]:
df['part_capital_etranger'].value_counts()

,count
part_capital_etranger,
0.0,16320
100.0,455
1000000.0,189
50.0,35
500000.0,9
...,...
2000000.0,1
700000.0,1
30.0,1


In [81]:
#verification des valeurs
(df['part_capital_etranger']<0).sum()

np.int64(0)

In [82]:
#verification du secteur avec le plus de capital etranger
df[['part_capital_etranger', 'Secteur']].value_counts()

part_capital_etranger  Secteur   
0.000000e+00           Tertiaire     14006
                       Secondaire     1745
                       Primaire        551
1.000000e+02           Tertiaire       343
1.000000e+06           Tertiaire       152
                                     ...  
1.000000e+09           Secondaire        1
2.065000e+09           Secondaire        1
9.068447e+09           Tertiaire         1
1.125667e+10           Secondaire        1
1.269362e+10           Tertiaire         1
Name: count, Length: 100, dtype: int64

In [83]:
# Somme des entreprises sans salarié
(df['salarie'] == 0).sum()

np.int64(168)

In [84]:
(df['salarie'] == 0).mean()*100
# ces entreprise reprensente 0.8 % du dataset

np.float64(0.8942353755256295)

In [85]:
(df['capital'] == 0).sum()
# Le nombre d'entreprises qui n'ont pas de capital predefinis

np.int64(11381)

**Exploration des 5 dernières variables**

Cette partie porte sur l'exploration des variables suivantes :

**salarie,nbre_empl_congolais,chiffre_aff_mo_annuel,chiffre_aff_mo_annuel_1, chiffre_aff_mo_annuel_2.**

L'objectif est compréhension de ces colonnes, le nettoyage et la qualité des données.

In [86]:
# Liste des dix premières valeurs des cinq dernières colonnes
df.iloc[:10, -5:]

,salarie,nbre_empl_congolais,chiffre_aff_mo_annuel,chiffre_aff_mo_annuel_1,chiffre_aff_mo_annuel_2
0,1.0,1.0,1000000,1500000,2000000.0
1,1.0,0.0,1500000,2000000,2500000.0
2,1.0,1.0,0,0,0.0
3,4.0,0.0,0,0,0.0
4,1.0,0.0,1500000,2000000,2500000.0
5,4.0,4.0,18000000,18000000,18000000.0
6,4.0,4.0,18000000,18000000,18000000.0
7,1.0,1.0,28000000,30000000,35000000.0
8,5.0,3.0,9395250,28091477,60000000.0
9,5.0,3.0,9395250,28091477,60000000.0


Changement de type de chaque variable

In [87]:

cols = ['salarie', 'nbre_empl_congolais',
        'chiffre_aff_mo_annuel', 'chiffre_aff_mo_annuel_1', 'chiffre_aff_mo_annuel_2']
df[cols].dtypes

,0
salarie,float64
nbre_empl_congolais,float64
chiffre_aff_mo_annuel,object
chiffre_aff_mo_annuel_1,int64
chiffre_aff_mo_annuel_2,float64


Vérifions d'abord les valeurs manquantes de ces cinq dernières colonnes.

Au regad des résultats ci-dessous, seules les variables salarie et nbre_empl_congolais qui ont respectivement 3 et 63 vides. Sur ce, nous allons considérer les variables salarie et nbre_empl_congolais en integer et les autres float.

In [88]:
df[cols].isna().sum()

,0
salarie,3
nbre_empl_congolais,63
chiffre_aff_mo_annuel,0
chiffre_aff_mo_annuel_1,0
chiffre_aff_mo_annuel_2,0


In [89]:
# Nettoyage des espaces (normaux et insécables) dans la colonne textuelle
df['chiffre_aff_mo_annuel'] = df['chiffre_aff_mo_annuel'].astype(str).str.replace(r'\s+', '', regex=True)

# Conversion des types
df = df.astype({
    'salarie': 'Int64',
    'nbre_empl_congolais': 'Int64',
    'chiffre_aff_mo_annuel': 'float64',
    'chiffre_aff_mo_annuel_1': 'float64',
    'chiffre_aff_mo_annuel_2': 'float64'
})

Analyse descriptive univariée sur les indicateurs de tendance centrale (moyenne, médiane et mode)

In [90]:
cols = df.columns[-5:]
print(cols.tolist())

# Version numérique de travail (df n'est pas modifié)
num = df[cols].apply(lambda s: pd.to_numeric(
    s.astype(str).str.replace(',', '.').str.replace(' ', ''), errors='coerce'))

num.info()

['salarie', 'nbre_empl_congolais', 'chiffre_aff_mo_annuel', 'chiffre_aff_mo_annuel_1', 'chiffre_aff_mo_annuel_2']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18787 entries, 0 to 18786
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   salarie                  18784 non-null  float64
 1   nbre_empl_congolais      18724 non-null  float64
 2   chiffre_aff_mo_annuel    18787 non-null  float64
 3   chiffre_aff_mo_annuel_1  18787 non-null  float64
 4   chiffre_aff_mo_annuel_2  18787 non-null  float64
dtypes: float64(5)
memory usage: 734.0 KB


**Visualisation des boites à moustaches pour détecter les valeurs abérrantes**

In [91]:
tendance = pd.DataFrame({
    'moyenne': num.mean(),
    'médiane': num.median(),
    'mode': num.mode().iloc[0]
})
tendance.round(2)

,moyenne,médiane,mode
salarie,2.198000e+01,1.0,1.0
nbre_empl_congolais,2.460000e+00,1.0,1.0
chiffre_aff_mo_annuel,1.010479e+08,2800000.0,1000000.0
chiffre_aff_mo_annuel_1,8.991517e+07,2500000.0,0.0
chiffre_aff_mo_annuel_2,9.798952e+07,3000000.0,0.0


In [92]:
cols = df.columns[-5:]
print(cols.tolist())

# Version numérique de travail (df n'est pas modifié)
num = df[cols].apply(lambda s: pd.to_numeric(
    s.astype(str).str.replace(',', '.').str.replace(' ', ''), errors='coerce'))

num.info()

['salarie', 'nbre_empl_congolais', 'chiffre_aff_mo_annuel', 'chiffre_aff_mo_annuel_1', 'chiffre_aff_mo_annuel_2']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18787 entries, 0 to 18786
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   salarie                  18784 non-null  float64
 1   nbre_empl_congolais      18724 non-null  float64
 2   chiffre_aff_mo_annuel    18787 non-null  float64
 3   chiffre_aff_mo_annuel_1  18787 non-null  float64
 4   chiffre_aff_mo_annuel_2  18787 non-null  float64
dtypes: float64(5)
memory usage: 734.0 KB


Ces graphiques nous montrent qu'il y a des valeurs abérrantes, les outliers qu'on peut éliminer pour éviter toutes mauvaises analyses. Ces outliers apparaissent sur les variables : salarie, et sur les chiffre_aff_mo_annuel 0 1 et 2. Ces outliers sont des valeurs isolées.